## 03 – Isochronen mit OpenRouteService (ORS)

Ziel: Fahrzeit-/Gehzeit-Isochronen für Apotheken in der Schweiz berechnen und auswerten.

**Kontext**
- Eingabe: `data/processed/pharmacies_ch.geojson` (EPSG:2056)
- Wir nutzen die **public ORS API** (Key erforderlich). Key gibt’s kostenlos bei: `https://openrouteservice.org/dev/#/signup`

Wichtig: ORS erwartet Koordinaten in **WGS84 (EPSG:4326)**. Wir transformieren daher von LV95 → WGS84 und nach der Abfrage wieder zurück nach LV95.

In [7]:
# 1) Imports + API Setup (openrouteservice oder requests direkt)

from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Any, Dict, Optional

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# Optional: openrouteservice SDK (falls installiert)
try:
    import openrouteservice  # type: ignore
    _HAS_ORS_SDK = True
except Exception:
    openrouteservice = None
    _HAS_ORS_SDK = False

import requests

PROCESSED = Path("..") / "data" / "processed"
OUTPUTS = Path("..") / "outputs" / "maps"
OUTPUTS.mkdir(parents=True, exist_ok=True)

# --- ORS Konfiguration (Public API) ---
# Du brauchst einen kostenlosen API-Key von https://openrouteservice.org/dev/#/signup
# WICHTIG: Keys gehören NICHT ins Repo. Nutze `.env` (lokal) oder echte Umgebungsvariablen.
ORS_BASE_URL = "https://api.openrouteservice.org"  # Vorgabe


def load_dotenv_key(dotenv_path: Path, key: str) -> str | None:
    """Minimaler .env-Loader (ohne zusätzliche Dependencies).

    Unterstützt u.a.:
    - ORS_API_KEY=...
    - export ORS_API_KEY=...
    - UTF-8 BOM am Dateianfang
    """
    if not dotenv_path.exists():
        return None
    try:
        for line in dotenv_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            if line.startswith("export "):
                line = line[len("export ") :].strip()
            k, v = line.split("=", 1)
            k = k.strip().lstrip("\ufeff")
            if k != key:
                continue
            v = v.strip().strip('"').strip("'")
            return v if v else None
    except Exception:
        return None
    return None


# 1) zuerst echte Env-Var, 2) sonst .env im Projekt-Root ("../.env" aus notebooks/)
DOTENV_PATH = (Path("..") / ".env").resolve()
ORS_API_KEY = os.getenv("ORS_API_KEY", "").strip() or (load_dotenv_key(DOTENV_PATH, "ORS_API_KEY") or "")

print("Setup OK")
print("- PROCESSED:", PROCESSED.resolve())
print("- OUTPUTS:", OUTPUTS.resolve())
print("- ORS_BASE_URL:", ORS_BASE_URL)
print("- .env gefunden:", DOTENV_PATH.exists(), "(", DOTENV_PATH, ")")
print("- ORS_API_KEY gesetzt?:", bool(ORS_API_KEY))
print("- ORS SDK installiert:", _HAS_ORS_SDK)

if not ORS_API_KEY:
    print("WARNUNG: ORS_API_KEY fehlt. Lege in `.env` z.B. an: ORS_API_KEY=... (oder setze env ORS_API_KEY)")


def _isochrones_endpoint(profile: str) -> str:
    # Public ORS v2 endpoint
    return f"{ORS_BASE_URL.rstrip('/')}/v2/isochrones/{profile}"


def ors_isochrones(
    profile: str,
    lon: float,
    lat: float,
    ranges_s: list[int],
    timeout_s: int = 90,
    retry: int = 2,
    sleep_s: float = 0.2,
) -> Dict[str, Any]:
    """Isochronen via ORS (SDK wenn vorhanden, sonst requests).

    - profile: z.B. "driving-car" oder "foot-walking"
    - lon/lat: WGS84
    - ranges_s: Sekunden
    """
    payload = {
        "locations": [[lon, lat]],
        "range": ranges_s,
        # optional: "interval" / "range_type"
        "range_type": "time",
    }

    headers: Dict[str, str] = {
        "Content-Type": "application/json",
        "Authorization": ORS_API_KEY,
    }

    # Wenn SDK vorhanden UND Key gesetzt: SDK nutzen (komfortabler)
    if _HAS_ORS_SDK and ORS_API_KEY:
        client = openrouteservice.Client(key=ORS_API_KEY)  # type: ignore
        res = client.isochrones(locations=[[lon, lat]], profile=profile, range=ranges_s, range_type="time")
        time.sleep(sleep_s)
        return res

    if not ORS_API_KEY:
        raise RuntimeError(
            "ORS_API_KEY fehlt. Bitte setze ihn als Umgebungsvariable ORS_API_KEY oder in ../.env (ORS_API_KEY=...)."
        )

    url = _isochrones_endpoint(profile)
    last_err: Optional[Exception] = None
    for attempt in range(retry + 1):
        try:
            r = requests.post(url, headers=headers, data=json.dumps(payload), timeout=timeout_s)
            if r.status_code >= 400:
                raise RuntimeError(f"ORS Fehler {r.status_code}: {r.text[:200]}")
            time.sleep(sleep_s)
            return r.json()
        except Exception as e:
            last_err = e
            backoff = 1.0 + attempt * 1.5
            print(f"Retry {attempt+1}/{retry} nach Fehler: {e}. Warte {backoff:.1f}s...")
            time.sleep(backoff)
    raise RuntimeError(f"ORS Anfrage gescheitert: {last_err}")


Setup OK
- PROCESSED: C:\Projects\pharmacy-accessibility-ch\data\processed
- OUTPUTS: C:\Projects\pharmacy-accessibility-ch\outputs\maps
- ORS_BASE_URL: https://api.openrouteservice.org
- .env gefunden: True ( C:\Projects\pharmacy-accessibility-ch\.env )
- ORS_API_KEY gesetzt?: False
- ORS SDK installiert: False
WARNUNG: ORS_API_KEY fehlt. Lege in `.env` z.B. an: ORS_API_KEY=... (oder setze env ORS_API_KEY)


In [8]:
# 2) Isochrone für eine einzelne Apotheke testen (5/10/15min zu Fuss + Auto)

pharm_path = PROCESSED / "pharmacies_ch.geojson"
gdf_pharm = gpd.read_file(pharm_path)
if gdf_pharm.crs is None:
    gdf_pharm = gdf_pharm.set_crs(2056)

print("Apotheken geladen:", len(gdf_pharm), "CRS:", gdf_pharm.crs)

# Nimm eine Apotheke (erste Zeile). ORS braucht WGS84 lon/lat.
row = gdf_pharm.iloc[0]
geom = row.geometry
pt_lv95 = geom if geom.geom_type == "Point" else geom.centroid
pt_wgs84 = gpd.GeoSeries([pt_lv95], crs=gdf_pharm.crs).to_crs(4326).iloc[0]
lon, lat = float(pt_wgs84.x), float(pt_wgs84.y)

ranges_s = [5 * 60, 10 * 60, 15 * 60]

print("Testpunkt WGS84:", (lon, lat))
print("Abfrage: foot-walking...")
res_walk = ors_isochrones("foot-walking", lon, lat, ranges_s)
print("OK – Features (walk):", len(res_walk.get("features", [])))

print("Abfrage: driving-car...")
res_car = ors_isochrones("driving-car", lon, lat, ranges_s)
print("OK – Features (car):", len(res_car.get("features", [])))

# Konvertiere zu GeoDataFrame für schnellen Plausibilitätscheck
gdf_walk = gpd.GeoDataFrame.from_features(res_walk["features"], crs=4326).to_crs(2056)
gdf_car = gpd.GeoDataFrame.from_features(res_car["features"], crs=4326).to_crs(2056)

print("Beispiel (walk) Spalten:", list(gdf_walk.columns))
print("Beispiel (car)  Spalten:", list(gdf_car.columns))
print("Fläche (car) in km² pro Range (nur Test):")
print((gdf_car.area / 1_000_000).round(2).to_string(index=False))


Apotheken geladen: 1640 CRS: EPSG:2056
Testpunkt WGS84: (8.726227708612216, 47.24097511006537)
Abfrage: foot-walking...


RuntimeError: ORS_API_KEY fehlt. Bitte setze ihn als Umgebungsvariable ORS_API_KEY oder in ../.env (ORS_API_KEY=...).

In [ ]:
# 3) Isochronen für alle Apotheken berechnen (5min Auto) → als GeoDataFrame
# Hinweis: Das sind bis zu 1640 API-Calls. Mit lokaler ORS-Instanz ok; in der Cloud ggf. Rate-Limits.
# Wir speichern inkrementell und können später "resume" machen.

out_gpkg = (PROCESSED / "isochrones_5min_car.gpkg").resolve()
layer_name = "isochrones_5min_car"

# ID-Spalte: wenn vorhanden osm_id nutzen, sonst Index
id_col = "osm_id" if "osm_id" in gdf_pharm.columns else None

already_done = set()
if out_gpkg.exists():
    try:
        gdf_existing = gpd.read_file(out_gpkg, layer=layer_name)
        if id_col and id_col in gdf_existing.columns:
            already_done = set(gdf_existing[id_col].astype(str).tolist())
        else:
            already_done = set(gdf_existing["pharmacy_id"].astype(str).tolist()) if "pharmacy_id" in gdf_existing.columns else set()
        print("Resume: vorhandene Isochronen geladen:", len(gdf_existing))
    except Exception as e:
        print("Konnte bestehende Datei nicht lesen (wird überschrieben):", e)
        out_gpkg.unlink(missing_ok=True)

ranges_s = [5 * 60]

features = []
t0 = time.time()
n_total = len(gdf_pharm)
n_new = 0

for i, r in gdf_pharm.iterrows():
    pharmacy_id = str(r[id_col]) if id_col else str(i)
    if pharmacy_id in already_done:
        continue

    geom = r.geometry
    pt_lv95 = geom if geom.geom_type == "Point" else geom.centroid
    pt_wgs84 = gpd.GeoSeries([pt_lv95], crs=gdf_pharm.crs).to_crs(4326).iloc[0]
    lon, lat = float(pt_wgs84.x), float(pt_wgs84.y)

    res = ors_isochrones("driving-car", lon, lat, ranges_s)

    # ORS liefert typ. genau 1 Feature für eine Range
    for f in res.get("features", []):
        f_props = f.get("properties", {})
        f_props["pharmacy_id"] = pharmacy_id
        if id_col:
            f_props[id_col] = r[id_col]
        f_props["profile"] = "driving-car"
        f_props["range_s"] = ranges_s[0]
        f["properties"] = f_props
        features.append(f)

    n_new += 1
    if n_new % 25 == 0:
        elapsed = time.time() - t0
        print(f"Berechnet: {n_new} neue Isochronen (von {n_total}). Laufzeit: {elapsed/60:.1f} min")

        gdf_chunk = gpd.GeoDataFrame.from_features(features, crs=4326).to_crs(2056)
        # Append: geopandas kann append in GPKG, indem wir existierende laden + concat (simpel, dafür robust)
        if out_gpkg.exists():
            gdf_prev = gpd.read_file(out_gpkg, layer=layer_name)
            gdf_all = pd.concat([gdf_prev, gdf_chunk], ignore_index=True)
        else:
            gdf_all = gdf_chunk
        gdf_all.to_file(out_gpkg, layer=layer_name, driver="GPKG")
        features = []

# Rest speichern
if features:
    gdf_chunk = gpd.GeoDataFrame.from_features(features, crs=4326).to_crs(2056)
    if out_gpkg.exists():
        gdf_prev = gpd.read_file(out_gpkg, layer=layer_name)
        gdf_all = pd.concat([gdf_prev, gdf_chunk], ignore_index=True)
    else:
        gdf_all = gdf_chunk
    gdf_all.to_file(out_gpkg, layer=layer_name, driver="GPKG")

# Ergebnis laden
gdf_iso_5_car = gpd.read_file(out_gpkg, layer=layer_name)
if gdf_iso_5_car.crs is None:
    gdf_iso_5_car = gdf_iso_5_car.set_crs(2056)

print("Alle 5min-Auto Isochronen verfügbar:", len(gdf_iso_5_car))
print("Gespeichert:", out_gpkg)
print("Beispiel-Fläche km² (erste 5):")
print(((gdf_iso_5_car.area / 1_000_000).head(5)).round(2).to_string(index=False))


In [ ]:
# 4) Isochronen-Union: Welche Fläche der Schweiz ist innerhalb 5/10/15min einer Apotheke?
# Idee: Wir berechnen für einen Test-Subset (oder alle) die Union der Isochronen und vergleichen mit CH-Fläche.
# Für CH-Fläche nutzen wir die Union der Gemeinden (gemeinden_ch.geojson) als Proxy.

gemeinden_path = PROCESSED / "gemeinden_ch.geojson"
gdf_gem = gpd.read_file(gemeinden_path)
if gdf_gem.crs is None:
    gdf_gem = gdf_gem.set_crs(2056)
gdf_gem = gdf_gem.to_crs(2056)

ch_geom = gdf_gem.geometry.unary_union
ch_area_km2 = ch_geom.area / 1_000_000

print("Gemeinden geladen:", len(gdf_gem))
print(f"CH-Fläche (Proxy via Gemeinden-Union): {ch_area_km2:,.1f} km²")

# Für 10/15 Minuten: wir nutzen die bereits funktionierende Einzel-Abfrage und rechnen die Union über Apotheken.
# Achtung: Das sind viele Calls. Hier als praktikabler Ansatz: 10/15min nur für Stichprobe ODER bei lokaler Instanz komplett.
# Public API hat Limits → default: Stichprobe, außer du setzt ORS_ALL=1
use_all = os.getenv("ORS_ALL", "0") == "1"
n = len(gdf_pharm) if use_all else min(150, len(gdf_pharm))

print("Berechne Union für", n, "Apotheken (ORS_ALL=1 → alle)")

def union_for_range(profile: str, range_s: int, n_rows: int) -> Any:
    polys = []
    for idx in range(n_rows):
        geom = gdf_pharm.iloc[idx].geometry
        pt_lv95 = geom if geom.geom_type == "Point" else geom.centroid
        pt_wgs84 = gpd.GeoSeries([pt_lv95], crs=gdf_pharm.crs).to_crs(4326).iloc[0]
        lon, lat = float(pt_wgs84.x), float(pt_wgs84.y)
        res = ors_isochrones(profile, lon, lat, [range_s])
        gdf_tmp = gpd.GeoDataFrame.from_features(res["features"], crs=4326).to_crs(2056)
        polys.extend(list(gdf_tmp.geometry))
        if (idx + 1) % 50 == 0:
            print(f"  {profile} {range_s//60}min: {idx+1}/{n_rows}")
    return gpd.GeoSeries(polys, crs=2056).unary_union


ranges = [5 * 60, 10 * 60, 15 * 60]
unions = []
for r_s in ranges:
    u = union_for_range("driving-car", r_s, n)
    area_km2 = u.area / 1_000_000
    pct = area_km2 / ch_area_km2 * 100
    unions.append({"profile": "driving-car", "minutes": r_s // 60, "area_km2": area_km2, "pct_CH": pct})
    print(f"Union driving-car {r_s//60}min: {area_km2:,.1f} km² ({pct:.1f}%)")

df_union = pd.DataFrame(unions)
print("\nUnion-Summary:")
print(df_union.to_string(index=False))


In [ ]:
# 5) Karte: Isochronen über Gemeindegrenzen, Apotheken als Punkte

gdf_gem_plot = gdf_gem.to_crs(2056)
gdf_pharm_plot = gdf_pharm.to_crs(2056)

# Falls Isochronen (5min car) nicht im Speicher sind, lade aus Datei
if "gdf_iso_5_car" not in globals():
    out_gpkg = (PROCESSED / "isochrones_5min_car.gpkg").resolve()
    gdf_iso_5_car = gpd.read_file(out_gpkg, layer="isochrones_5min_car")
    if gdf_iso_5_car.crs is None:
        gdf_iso_5_car = gdf_iso_5_car.set_crs(2056)

out_png = (OUTPUTS / "03_isochronen_5min_auto.png").resolve()

fig, ax = plt.subplots(figsize=(14, 10))
gdf_gem_plot.boundary.plot(ax=ax, linewidth=0.25, color="#666666", alpha=0.6)

# Isochronen (5min Auto)
gdf_iso_5_car.plot(ax=ax, color="#ff6b6b", alpha=0.18, edgecolor="none")

# Apothekenpunkte
gdf_pharm_plot.plot(ax=ax, color="#1f3a93", markersize=3, alpha=0.7)

ax.set_title("Isochronen 5 Minuten (Auto) zu Apotheken", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(out_png, dpi=200, bbox_inches="tight")
plt.show()

print("Karte gespeichert:", out_png)
print("Layer:")
print("- Gemeinden:", len(gdf_gem_plot))
print("- Apotheken:", len(gdf_pharm_plot))
print("- Isochronen 5min Auto:", len(gdf_iso_5_car))


In [ ]:
# 6) Ergebnisse speichern in data/processed/

# Speichere Isochronen (5min Auto) zusätzlich als GeoJSON (handlich, aber größer)
out_geojson = (PROCESSED / "isochrones_5min_car.geojson").resolve()
try:
    gdf_iso_5_car.to_file(out_geojson, driver="GeoJSON")
    print("Isochronen GeoJSON gespeichert:", out_geojson)
except Exception as e:
    print("Konnte GeoJSON nicht schreiben (evtl. Dateisperre/Encoding):", e)

# Speichere Union-Summary (falls vorhanden)
out_union_csv = (PROCESSED / "isochronen_union_summary.csv").resolve()
if "df_union" in globals():
    df_union.to_csv(out_union_csv, index=False)
    print("Union-Summary CSV gespeichert:", out_union_csv)
else:
    print("Union-Summary nicht vorhanden (Zelle 4 ggf. noch nicht ausgeführt).")

print("Fertig – Outputs liegen in data/processed/ und outputs/maps/")
